### OpenAI Assistants API
https://platform.openai.com/docs/assistants/overview

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

assistant = client.beta.assistants.create(
  name="Math Tutor",
  instructions="You are a personal math tutor. Write and run code to answer math questions.",
  tools=[{"type": "code_interpreter"}],
  model="gpt-4o",
)

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
# 假设您已经将 Assistant ID 保存在了某个地方
assistant_id = "asst_tne8b2j2bWMGhgDecbyQ5eFy" # 这是您之前创建并保存的 ID

# 使用 retrieve 方法来获取 Assistant 对象
assistant = client.beta.assistants.retrieve(
  assistant_id=assistant_id
)

print(f"成功获取已存在的 Assistant: {assistant.name} (ID: {assistant.id})")

成功获取已存在的 Assistant: Math Tutor (ID: asst_tne8b2j2bWMGhgDecbyQ5eFy)


In [9]:
thread = client.beta.threads.create()

/var/folders/9c/cd2526rs26sbpw08vr1ycl1r0000gn/T/ipykernel_10340/1326757544.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create()


In [10]:
client.beta.threads.messages.create(
  thread_id=thread.id,
  role="user",
  content="I need to solve the equation `3x + 11 = 14`. Can you help me?"
)

/var/folders/9c/cd2526rs26sbpw08vr1ycl1r0000gn/T/ipykernel_10340/2445994090.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  client.beta.threads.messages.create(


Message(id='msg_m9v3wnc34BDuc5cXDDok7og9', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='I need to solve the equation `3x + 11 = 14`. Can you help me?'), type='text')], created_at=1752290164, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_6uuUMLIX0SF5JZpKmjp6b8mc')

In [11]:
from typing_extensions import override
from openai import AssistantEventHandler
 
# First, we create a EventHandler class to define
# how we want to handle the events in the response stream.
 
class EventHandler(AssistantEventHandler):    
  @override
  def on_text_created(self, text) -> None:
    print(f"\nassistant > ", end="", flush=True)
      
  @override
  def on_text_delta(self, delta, snapshot):
    print(delta.value, end="", flush=True)
      
  def on_tool_call_created(self, tool_call):
    print(f"\nassistant > {tool_call.type}\n", flush=True)
  
  def on_tool_call_delta(self, delta, snapshot):
    if delta.type == 'code_interpreter':
      if delta.code_interpreter.input:
        print(delta.code_interpreter.input, end="", flush=True)
      if delta.code_interpreter.outputs:
        print(f"\n\noutput >", flush=True)
        for output in delta.code_interpreter.outputs:
          if output.type == "logs":
            print(f"\n{output.logs}", flush=True)
 
# Then, we use the `stream` SDK helper 
# with the `EventHandler` class to create the Run 
# and stream the response.
 
with client.beta.threads.runs.stream(
  thread_id=thread.id,
  assistant_id=assistant.id,
  instructions="Please address the user as Jane Doe. The user has a premium account.",
  event_handler=EventHandler(),
) as stream:
  stream.until_done()

/var/folders/9c/cd2526rs26sbpw08vr1ycl1r0000gn/T/ipykernel_10340/910384089.py:33: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  with client.beta.threads.runs.stream(



assistant > code_interpreter

import sympy as sp

# Define the variable
x = sp.symbols('x')

# Define the equation
equation = 3*x + 11 - 14

# Solve the equation
solution = sp.solve(equation, x)
solution
assistant > The solution to the equation \(3x + 11 = 14\) is \(x = 1\).

In [3]:
# 查询Assistant
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Initialize the OpenAI client
# The client automatically reads the OPENAI_API_KEY from the environment
client = OpenAI()

try:
    # Fetch the list of all assistants
    my_assistants = client.beta.assistants.list(
        order="desc",
        limit="20", # You can adjust the limit as needed
    )

    # Check if any assistants were found
    if not my_assistants.data:
        print("🤖 No assistants found for this API key.")
    else:
        print("🤖 Found assistants for this API key:\n")
        # Iterate through the assistants and print their details
        for assistant in my_assistants.data:
            print(f"Name: {assistant.name}")
            print(f"ID: {assistant.id}\n" + "-"*30)

except Exception as e:
    print(f"An error occurred: {e}")

🤖 Found assistants for this API key:

Name: Chatterbox - Generate
ID: asst_52vwJ23aPmmZ0SHWvkJjwSQ8
------------------------------
Name: Chatterbox
ID: asst_BtPmdAAlarvt1RsT8WAB75Gc
------------------------------


In [2]:
# 删除Assistant
import os
from dotenv import load_dotenv
from openai import OpenAI

# 从 .env 文件加载环境变量
load_dotenv()

# 初始化 OpenAI 客户端
# 客户端会自动从环境中读取 OPENAI_API_KEY
client = OpenAI()

# --- 请将这里替换成您想要删除的 Assistant 的 ID ---
# 您可以从创建 Assistant 时的返回结果、列出 Assistant 的脚本或 OpenAI 网站的 Assistants 页面找到这个 ID
assistant_id_to_delete = "asst_tne8b2j2bWMGhgDecbyQ5eFy" 

# 确保 assistant_id 不是默认的占位符
if assistant_id_to_delete == "asst_xxxxxxxxxxxxxxxxxxxxxxxx":
    print("❌ 错误：请将 'assistant_id_to_delete' 变量替换为您要删除的 Assistant 的实际 ID。")
else:
    try:
        print(f"正在尝试删除 Assistant: {assistant_id_to_delete}...")
        
        # 调用 API 删除指定的 Assistant
        response = client.beta.assistants.delete(
          assistant_id=assistant_id_to_delete
        )

        # 检查返回结果以确认删除成功
        # 成功删除后，response.deleted 会是 True
        if response.deleted:
            print(f"✅ 成功删除 Assistant: {response.id}")
        else:
            print("🤔 删除请求已发送，但未收到成功确认。请检查 OpenAI 控制台。")

    except Exception as e:
        # 捕获并打印可能发生的错误（例如，ID 不存在、权限问题等）
        print(f"❌ 删除失败，发生错误: {e}")



正在尝试删除 Assistant: asst_tne8b2j2bWMGhgDecbyQ5eFy...
✅ 成功删除 Assistant: asst_tne8b2j2bWMGhgDecbyQ5eFy
